In [1]:
!pip install -q timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.6 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
import os
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
import timm

import numpy as np
import torch
import torch.nn as nn

In [3]:
model = timm.create_model(
    "vit_base_patch14_dinov2",
    pretrained=True,
    num_classes=0,
    dynamic_img_size=True
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model.eval()
model.to(DEVICE)
print(DEVICE)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

cuda


In [10]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

transform = A.Compose([
    A.PadIfNeeded(
        min_height=518,
        min_width=518,
        border_mode=0,
        fill=(0, 0, 0),
        position="center"
    ),
    A.Resize(518, 518),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

img_path = r"/kaggle/input/datasets/iowiqo/lab666/lab6/inaction/1/im1.jpg"
image = cv2.imread(str(img_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

x = transform(image=image)["image"]
x = x.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    emb = model(x)

print(emb.shape)

torch.Size([1, 768])


In [13]:
# ============================================================
# CONFIG
# ============================================================

# Корень исходного датасета
DATA_ROOT = Path("/kaggle/input/datasets/iowiqo/lab666/lab6")

# Куда сохранять эмбеддинги
OUTPUT_ROOT = Path("/kaggle/working/embeddings")

# ============================================================
# COLLECT IMAGES
# ============================================================

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

all_images = []

for path in DATA_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        all_images.append(path)

print(f"Found {len(all_images)} images")

# ============================================================
# FEATURE EXTRACTION
# ============================================================


@torch.no_grad()
def extract_embedding(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    x = transform(image=image)["image"]
    x = x.unsqueeze(0).to(DEVICE)

    emb = model(x)

    return emb.squeeze(0).cpu().numpy().astype(np.float32)

# ============================================================
# MAIN LOOP
# ============================================================

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for image_path in tqdm(all_images):

    # относительный путь внутри датасета
    rel_path = image_path.relative_to(DATA_ROOT)

    # work/track_001/000123.jpg
    # ->
    # embeddings/work/track_001/000123.npy

    save_path = (
        OUTPUT_ROOT /
        rel_path
    ).with_suffix(".npy")

    save_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # пропускаем уже обработанные
    if save_path.exists():
        continue

    try:
        emb = extract_embedding(image_path)
        np.save(save_path, emb)

    except Exception as e:
        print(f"\nERROR: {image_path}")
        print(e)

print("\nDone.")
print(f"Saved to: {OUTPUT_ROOT}")

Found 24237 images


  0%|          | 0/24237 [00:00<?, ?it/s]


Done.
Saved to: /kaggle/working/embeddings
